In [6]:
reset()

import numpy as np
import networkx as nx
from numpy import linalg as LA

# if for some reason the target is not in the span, sagemath will raise an arithmetic error
def find_linear_combination(basis, target):
    dim = len(basis) + 1  # we want the len of the superbasis
    basis_sage = []
    basis_sage.append([vector(x) for x in basis])
    basis_sage = basis_sage[0]  # we ignore the superbasis vector!!!
    K = QQ  # K is the rational field
    VS = (K ** (dim)).span_of_basis(basis_sage)  # we define the subspace generated by v[0],v[1],..,v[n-2]

    try:
        z = VS.coordinate_vector(vector(target))
        z = vector(list(z) + [0])
        return z
    except ArithmeticError:
        print("Oops! vector is not in the span.")
        return


def create_weighted_graph_networkx(dimension, S, Q):
    """
    source and sink are two positive integers, source = 0 and sink = n+2
    S=(s_i) is given by s_i=-2sum_{j} q_{ij}p_j
    Graph has n+3 nodes: the 1st node is the source = 0, the next n+1 nodes represent each vector 
    of the superbasis and thelast n+3 th node is the sink
    """
    graph = nx.Graph()
    source = 0
    sink = dimension + 2
    graph.add_nodes_from(range(source, sink))
    graph.add_edge(source, sink, capacity=0.0)
    
        
    for i in range(1, dimension+2):
        for j in range(i+1, dimension+2):
            node1 = i
            node2 = j
            graph.add_edge(node1, node2, capacity=-Q[i-1][j-1])
    
    for i in range(0, dimension+1):
        node = i+1
        if S[i] < 0:
            graph.add_edge(source, node, capacity=-S[i])
            graph.add_edge(node, sink, capacity=0.0)
        elif S[i] >= 0:
            graph.add_edge(source, node, capacity=0.0)
            graph.add_edge(node, sink, capacity=S[i])
    # print("EDGES: ", graph.edges.data())
    # print(laplacian_matrix(graph))
    return graph

def get_qij(superbasis):
    n = len(superbasis)
    Q = [superbasis[i].dot_product(superbasis[j]) for i in range(n) for j in range(n)]
    Q = matrix(n, n, Q)
    return Q

def get_superbasis(basis): # basis is a list
    n = len(basis)
    basis = [vector(x) for x in basis]
    superbasis_vector = -sum(basis[i] for i in [0..n-1])
    return [vector(basis[i]) for i in [0..n-1]] + [vector(superbasis_vector)]

def vfk_root_lattice(N): 
    # returns the superbasis for the root lattice An
    M = []
    L = [0]
    L1 = [-1, 1]
    for i in range(N - 1):
        M.append(L * i + L1 + L * (N - 2 - i))
    M = [np.array(x).astype(int) for x in M]
    v0 = -sum(M[i] for i in range(N - 1))
    M.append(v0)
    M=[vector(x) for x in M]
    return M

def get_qij(superbasis):
    n = len(superbasis)
    Q = [superbasis[i].dot_product(superbasis[j]) for i in range(n) for j in range(n)]
    Q = matrix(n, n, Q)
    return Q

def get_superbasis(basis): # basis is a list
    n = len(basis)
    basis = [vector(x) for x in basis]
    superbasis_vector = -sum(basis[i] for i in [0..n-1])
    return [vector(basis[i]) for i in [0..n-1]] + [vector(superbasis_vector)]

def calculate_s(z, u, Q):
    s = []
    p = np.subtract(z, u)
    rows,columns=Q.dimensions()
    for i in range(rows):
        elements = []
        for j in range(columns):
            elements.append(Q[i][j] * p[j])
        s.append(-2 * np.sum(elements))
    return s

def length(v):
    n = len(v[0])
    x=0
    print(n)
    for i in range(n):
        x=x+v[0][i]**2
    return x.expand()

def t_coeffs(dimension, set_c, source):
    set_c.remove(source)
    t = np.zeros(shape=dimension)
    for i in set_c:
        t[i-1] = 1
    return t

def do_mincut(n,S,Q):
    G = create_weighted_graph_networkx(n,S,Q) # S depends on the target vector.
    source=0
    sink=n+2
    cut_value, partition = nx.minimum_cut(G, 0, sink)
    reachable, non_reachable = partition
    cut = reachable if source in reachable else non_reachable
    t_c = t_coeffs(n+1,cut,0)
    return t_c

def cvp_sage(target_vector,basis):   
    # Works well only il low dimensions
    from fpylll import IntegerMatrix, CVP, LLL
    from sage.modules.free_module_integer import IntegerLattice
    L = IntegerLattice(matrix(basis))
    L.LLL()
    v = L.closest_vector(target_vector)
    print("CVP(L,target) result from sagemath:", v)
    distance = LA.norm(np.subtract(target_vector, v))
    print("distance=",distance)
    return 


Given ${\bf z}$ and ${\bf u}_k$ we have,
the matrix $Q$ is symmetric, $Q=(q_{ij})=({\bf b}_i\cdot {\bf b}_j)$ for $1\le i,j\le n+1$ and the vector ${\bf s}$ is ${\bf s}=(s_i)$ given by $s_i=-2\sum_{j} q_{ij}p_j$ where $q_{ij}={\bf b}_i\cdot {\bf b}_j$ and ${\bf p}=(p_j)={\bf z}-{\bf u}_k.$

In [7]:
n=40
super_basis = vfk_root_lattice(n+1)
basis=[x for x in super_basis[0:n]]
#print(basis)
#target_vector = [-17/4, 9/4, 0, 2]
target_vector = 1/3*super_basis[0]+25*super_basis[2]+1/6*super_basis[n]
Q=get_qij(super_basis)
z = find_linear_combination(basis, target_vector)
#print("z=",z)
# the previous are permanent


In [8]:
u_point = np.floor(np.array(z))
p = z - u_point
#print("u=",u_point)
#print("p=z-u=",p)
S = calculate_s(z,u_point,Q)
#print("S=",S)
vec = np.dot(u_point, super_basis)
print("x=",vec)
print(LA.norm(np.subtract(target_vector, vec)))

x= [  0.   1. -25.  25.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.  -1.]
1.0801234497346435


In [9]:
# def (n,S,Q,super_basis,target_vector,z)
for i in range(n):  # the iter should converge at n iters, not n+1
    # TODO: add if y (target) is inside the voronoi cell of u then stop the procedure.
    distance_old = LA.norm(np.subtract(target_vector, vec))
    print("++++++++++++++++++++++++++++++++++++++++++++++++++")
    print("Iter: ", i)
    t = do_mincut(n,S,Q) # you must update S
    u_point = np.add(u_point, t)
    vec = np.dot(u_point, super_basis)
    #print("x=",vec)
    distance = LA.norm(np.subtract(target_vector, vec))
    if distance>=distance_old:
        print("vector=",vec)
        print("distance=",distance)
        print("+++++++")
        break
    #print("t=",t)
    S = calculate_s(z,u_point,Q)
    #print("S=",S)
    #print("u_point",u_point)
    #print("distance=",distance)
    


++++++++++++++++++++++++++++++++++++++++++++++++++
Iter:  0
++++++++++++++++++++++++++++++++++++++++++++++++++
Iter:  1
vector= [  0.   0. -25.  25.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
distance= 0.408248290463863
+++++++
